# Problema 1 — Evidencia para decisiones de datos

Esta libreta **no decide** qué hacer con los códigos raros ni con `PAY_*`.
Solo produce la evidencia descriptiva y las métricas de validación cruzada
para que el alumno tome la decisión.

Reglas que respeta la libreta:

- El 70% de entrenamiento se usa para validar; el 30% de prueba **no se toca**.
- La semilla común se importa desde `common.py` (`SEED = 223210350`).
- Todas las transformaciones de columnas van dentro de un `Pipeline`.
- No se hace búsqueda de hiperparámetros.
- No se elige una variante como recomendación final.

In [1]:
"""Configuración, rutas e importaciones de la libreta."""

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _buscar_raiz(inicio: Path) -> Path:
    """Sube desde el directorio actual hasta encontrar la raíz del repo.

    La raíz se reconoce porque contiene `common.py` y la carpeta `problema-1/`.
    Así la libreta funciona igual si se abre desde la raíz o desde
    `problema-1/`, y también cuando la ejecuta `nbconvert`.
    """
    for base in [inicio, *inicio.parents]:
        if (base / "common.py").exists() and (base / "problema-1").is_dir():
            return base
    return inicio


_AQUI = Path.cwd()
_RAIZ = _buscar_raiz(_AQUI)
_DIR_PROBLEMA = _RAIZ / "problema-1" if (_RAIZ / "problema-1").is_dir() else _RAIZ

# Permite importar common.py (raíz) y los módulos de problema-1/.
for _ruta in (_RAIZ, _DIR_PROBLEMA):
    if str(_ruta) not in sys.path:
        sys.path.insert(0, str(_ruta))

from common import SEED
from data import TARGET, cargar_dataframe
import evidencia_datos as ev

RESULTADOS = _RAIZ / "resultados"
RESULTADOS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

print(f"SEED = {SEED}")
print(f"Variable objetivo = '{TARGET}'")
print(f"Directorio de resultados = {RESULTADOS}")

SEED = 223210350
Variable objetivo = 'default payment next month'
Directorio de resultados = /home/tito/repos/RDP/Examen_1_RDP/resultados


## Carga de datos

`cargar_dataframe()` descarga el archivo de UCI si no existe y corrige el
encabezado desplazado. Aquí solo verificamos dimensiones y columnas PAY_*
disponibles; no se imputa ni se recodifica ningún valor.

In [2]:
df = cargar_dataframe()
columnas_pay = ev.obtener_columnas_pay(df)

# ID es un identificador y la variable objetivo no puede ser predictora.
FEATURES = [c for c in df.columns if c not in ("ID", TARGET)]

print(f"Dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Columnas PAY_* disponibles: {columnas_pay}")
print(f"Variables de entrada ({len(FEATURES)}): {FEATURES}")
print(f"Distribución del objetivo: {df[TARGET].value_counts().to_dict()}")

Dataset: 30000 filas x 25 columnas
Columnas PAY_* disponibles: ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
Variables de entrada (23): ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Distribución del objetivo: {0: 23364, 1: 6636}


## 1. Descriptivo de códigos raros y PAY_0

Cada tabla muestra, por código: `n` (cuántas filas tienen ese código), la
tasa de incumplimiento y la composición por sexo (`n_hombres`, `n_mujeres`,
`pct_hombres`, `pct_mujeres`). La columna `en_descripcion_oficial` marca
los códigos que el enunciado señala como no documentados. Después de cada
tabla hay un comentario con su lectura.

In [3]:
print("EDUCATION — tabla por código")
tabla_education = ev.tabla_descriptiva(
    df, "EDUCATION", ev.CODIGOS_RAROS["EDUCATION"], TARGET
)
display(tabla_education)

EDUCATION — tabla por código


,variable,codigo,en_descripcion_oficial,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,EDUCATION,0,no,14,8,6,57.142857,42.857143,0.000000
1,EDUCATION,1,sí,10585,4354,6231,41.133680,58.866320,19.234766
2,EDUCATION,2,sí,14030,5374,8656,38.303635,61.696365,23.734854
3,EDUCATION,3,sí,4917,1990,2927,40.471832,59.528168,25.157616
4,EDUCATION,4,sí,123,42,81,34.146341,65.853659,5.691057
5,EDUCATION,5,no,280,95,185,33.928571,66.071429,6.428571
6,EDUCATION,6,no,51,25,26,49.019608,50.980392,15.686275


En `EDUCATION`, los códigos 0, 5 y 6 son los no documentados. La tabla
permite ver cuántas filas tiene cada uno y si su tasa de incumplimiento se
separa de la de los códigos 1–4. La composición por sexo ayuda a detectar si
esos códigos se concentran en hombres o en mujeres; con `n` pequeños, las
tasas son más inestables.

In [4]:
print("EDUCATION — comparación documentados vs. no documentados")
display(
    ev.tabla_comparacion(
        df, "EDUCATION", ev.CODIGOS_RAROS["EDUCATION"], TARGET
    )
)

EDUCATION — comparación documentados vs. no documentados


,variable,grupo,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,EDUCATION,documentado,29655,11760,17895,39.656045,60.343955,22.289664
1,EDUCATION,no documentado,345,128,217,37.101449,62.898551,7.536232


La comparación agrupa todos los códigos no documentados de `EDUCATION` en una
sola fila y todos los documentados en otra. Si el grupo no documentado tiene
una tasa parecida a la documentada, agruparlo o conservarlo como categoría
propia no cambiaría mucho el patrón descriptivo; si la tasa es muy distinta,
la decisión sí puede afectar al modelo.

In [5]:
print("MARRIAGE — tabla por código")
tabla_marriage = ev.tabla_descriptiva(
    df, "MARRIAGE", ev.CODIGOS_RAROS["MARRIAGE"], TARGET
)
display(tabla_marriage)

MARRIAGE — tabla por código


,variable,codigo,en_descripcion_oficial,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,MARRIAGE,0,no,54,14,40,25.925926,74.074074,9.259259
1,MARRIAGE,1,sí,13659,5190,8469,37.996925,62.003075,23.471704
2,MARRIAGE,2,sí,15964,6553,9411,41.048609,58.951391,20.928339
3,MARRIAGE,3,sí,323,131,192,40.557276,59.442724,26.006192


En `MARRIAGE`, el código no documentado es el 0. La tabla permite comparar
su frecuencia, su tasa de incumplimiento y su mezcla por sexo contra los
códigos 1–3. Si el código 0 tiene pocas filas, cualquier diferencia en la
tasa debe leerse con cautela.

In [6]:
print("MARRIAGE — comparación documentados vs. no documentados")
display(
    ev.tabla_comparacion(
        df, "MARRIAGE", ev.CODIGOS_RAROS["MARRIAGE"], TARGET
    )
)

MARRIAGE — comparación documentados vs. no documentados


,variable,grupo,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,MARRIAGE,documentado,29946,11874,18072,39.651372,60.348628,22.143191
1,MARRIAGE,no documentado,54,14,40,25.925926,74.074074,9.259259


La comparación documentados vs. no documentados de `MARRIAGE` resume cuánto
pesa el código 0 en la muestra y si su tasa de incumplimiento se aleja de la
de los códigos oficiales. Es evidencia descriptiva; no decide si el código 0
debe agruparse, eliminarse o mantenerse como categoría propia.

In [7]:
print("PAY_0 — tabla por valor")
tabla_pay_0 = ev.tabla_descriptiva(
    df, "PAY_0", ev.CODIGOS_RAROS_PAY_0, TARGET
)
display(tabla_pay_0)

PAY_0 — tabla por valor


,variable,codigo,en_descripcion_oficial,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,PAY_0,-2,no,2759,871,1888,31.569409,68.430591,13.229431
1,PAY_0,-1,sí,5686,2132,3554,37.495603,62.504397,16.778051
2,PAY_0,0,no,14737,6031,8706,40.924204,59.075796,12.811291
3,PAY_0,1,sí,3688,1492,2196,40.455531,59.544469,33.947939
4,PAY_0,2,sí,2667,1120,1547,41.994751,58.005249,69.141357
5,PAY_0,3,sí,322,159,163,49.378882,50.621118,75.776398
6,PAY_0,4,sí,76,44,32,57.894737,42.105263,68.421053
7,PAY_0,5,sí,26,18,8,69.230769,30.769231,50.000000
8,PAY_0,6,sí,11,6,5,54.545455,45.454545,54.545455
9,PAY_0,7,sí,9,5,4,55.555556,44.444444,77.777778


En `PAY_0`, los valores no documentados son `-2` y `0`. La tabla muestra
todos los valores observados, incluidos los meses de atraso documentados
(`-1` y `1` en adelante). Así se puede comparar si los estados no
documentados tienen una tasa de incumplimiento distinta y si su proporción
de hombres y mujeres cambia respecto a los estados documentados.

In [8]:
print("PAY_0 — comparación documentados vs. no documentados")
display(
    ev.tabla_comparacion(
        df, "PAY_0", ev.CODIGOS_RAROS_PAY_0, TARGET
    )
)

PAY_0 — comparación documentados vs. no documentados


,variable,grupo,n,n_hombres,n_mujeres,pct_hombres,pct_mujeres,tasa_incumplimiento
0,PAY_0,documentado,12504,4986,7518,39.875240,60.124760,35.052783
1,PAY_0,no documentado,17496,6902,10594,39.449017,60.550983,12.877229


La comparación de `PAY_0` junta los estados `-2` y `0` y los contrasta con el
resto de los valores. Una diferencia grande de tasa sugiere que el
tratamiento de `PAY_0` (numérico tal cual vs. indicadores de estado) puede
importar; una diferencia pequeña sugiere que ambas variantes describen
patrones parecidos.

## 2. Split estratificado 70/30

Se estratifica por la variable objetivo para conservar la proporción de
incumplimientos. Los índices se guardan en
`resultados/split_indices.csv`. A partir de este punto, la libreta solo usa
el 70% de entrenamiento: el 30% de prueba no se consulta.

In [9]:
idx_train, idx_test = ev.dividir_estratificado(
    df, TARGET, test_size=0.30, seed=SEED
)
tabla_indices = ev.guardar_indices(
    df,
    idx_train,
    idx_test,
    TARGET,
    RESULTADOS / "split_indices.csv",
)

X_train = df.iloc[idx_train][FEATURES].reset_index(drop=True)
y_train = df.iloc[idx_train][TARGET].reset_index(drop=True)

print(f"Entrenamiento: {len(idx_train)} filas")
print(f"Prueba reservada: {len(idx_test)} filas (no se usa en esta libreta)")
print(f"Tasa de incumplimiento en entrenamiento: {y_train.mean():.4%}")
print(f"Archivo de índices: {RESULTADOS / 'split_indices.csv'}")
display(tabla_indices["conjunto"].value_counts().to_frame("filas"))

Entrenamiento: 21000 filas
Prueba reservada: 9000 filas (no se usa en esta libreta)
Tasa de incumplimiento en entrenamiento: 22.1190%
Archivo de índices: /home/tito/repos/RDP/Examen_1_RDP/resultados/split_indices.csv


,filas
conjunto,
train,21000
test,9000


La tabla confirma que el 70% quedó etiquetado como `train` y el 30% como
`test`. Esa columna `test` solo se guarda como referencia; en las secciones
siguientes no se calcula ninguna métrica sobre ella.

## 3. Significado de códigos no documentados

Esta sección usa **solo el 70% de entrenamiento** (`X_train`/`y_train`).
No recodifica, imputa ni agrupa ningún valor: solo describe qué filas tienen
cada código y cómo se relacionan con facturación, pago, edad, límite de
crédito y sexo. Las tasas y los intervalos se muestran en porcentaje. El
intervalo de Wilson se usa porque es más estable con muestras chicas que la
aproximación normal.

El dataset no incluye un diccionario oficial de los códigos no documentados,
así que estas tablas no permiten afirmar qué "significa" cada uno.

In [10]:
# Vista del 70% de entrenamiento con la variable objetivo incluida.
# No se recodifica nada y no se consulta el 30% de prueba.
df_train = X_train.copy()
df_train[TARGET] = y_train

In [11]:
print("PAY_0 — códigos -2, -1, 0, 1, 2 (solo entrenamiento)")
tabla_significado_pay_0 = ev.tabla_pay_significado(
    df_train,
    columna_pay="PAY_0",
    columna_bill="BILL_AMT1",
    columna_pay_amt="PAY_AMT1",
    objetivo=TARGET,
)
display(tabla_significado_pay_0)

ruta_significado_pay_0 = RESULTADOS / "significado_pay_0.csv"
tabla_significado_pay_0.to_csv(
    ruta_significado_pay_0, index=False, float_format="%.6f"
)
print(f"Guardado en: {ruta_significado_pay_0}")

PAY_0 — códigos -2, -1, 0, 1, 2 (solo entrenamiento)


,variable_pay,codigo,n,tasa_incumplimiento,pct_bill_no_positivo,mediana_bill,mediana_pay_amt,n_bill_positivo,mediana_ratio_pay_amt_bill
0,PAY_0,-2,1913,12.859383,31.677993,1289.0,1131.0,1307,0.813225
1,PAY_0,-1,3985,17.114178,0.100376,2711.0,1661.0,3981,0.774554
2,PAY_0,0,10308,12.514552,0.000000,50094.0,3000.0,10308,0.052684
3,PAY_0,1,2599,33.897653,46.094652,4473.0,0.0,1401,0.000000
4,PAY_0,2,1850,69.891892,0.000000,41282.5,2000.0,1850,0.048766


Guardado en: /home/tito/repos/RDP/Examen_1_RDP/resultados/significado_pay_0.csv


**Qué sugiere.** Los códigos `-2`, `-1` y `0` tienen tasas de
incumplimiento menores (12.9%, 17.1% y 12.5%) que `1` y `2` (33.9% y
69.9%). También cambia el perfil de facturación y pago: `0` tiene una
mediana de `BILL_AMT1` de 50,094 y casi ninguna factura no positiva; `-2`
tiene 31.7% de facturas no positivas y una mediana de 1,289; `1` tiene 46.1%
de facturas no positivas y mediana de `PAY_AMT1` igual a 0; `2` tiene una
mediana de factura de 41,282.5 y una razón pago/factura cercana a 0.05. Eso
es compatible con que los códigos negativos/cero distingan estados de
facturación o de atraso, pero es solo una lectura descriptiva.

**Qué NO se puede concluir.** No hay un diccionario oficial que explique los
códigos no documentados (`-2` y `0`), así que no se puede afirmar que `-2`
sea "sin consumo" o que `0` sea "sin atraso" por definición. Las diferencias
son asociaciones, no efectos causales; además, `PAY_0` y las
facturas/pagos provienen del mismo historial, por lo que parte de la
relación es mecánica. Los códigos `1` y `2` tienen menos filas que `0`, y
la razón pago/factura solo usa las filas con factura positiva.

In [12]:
print("PAY_2 — verificación con BILL_AMT2 y PAY_AMT2 (solo entrenamiento)")
tabla_significado_pay_2 = ev.tabla_pay_significado(
    df_train,
    columna_pay="PAY_2",
    columna_bill="BILL_AMT2",
    columna_pay_amt="PAY_AMT2",
    objetivo=TARGET,
)
display(tabla_significado_pay_2)

ruta_significado_pay_2 = RESULTADOS / "significado_pay_2.csv"
tabla_significado_pay_2.to_csv(
    ruta_significado_pay_2, index=False, float_format="%.6f"
)
print(f"Guardado en: {ruta_significado_pay_2}")

PAY_2 — verificación con BILL_AMT2 y PAY_AMT2 (solo entrenamiento)


,variable_pay,codigo,n,tasa_incumplimiento,pct_bill_no_positivo,mediana_bill,mediana_pay_amt,n_bill_positivo,mediana_ratio_pay_amt_bill
0,PAY_2,-2,2659,17.976683,55.396766,0.0,181.0,1186,0.866064
1,PAY_2,-1,4211,16.409404,11.612444,1989.0,1599.0,3722,0.957273
2,PAY_2,0,11016,15.686275,1.797386,48703.5,2863.5,10818,0.050234
3,PAY_2,1,19,15.789474,73.684211,0.0,2.0,5,0.332802
4,PAY_2,2,2750,55.854545,1.745455,27704.0,1621.5,2702,0.045537


Guardado en: /home/tito/repos/RDP/Examen_1_RDP/resultados/significado_pay_2.csv


**Qué sugiere.** La verificación con `PAY_2` repite parte del patrón:
`-2`, `-1` y `0` tienen tasas de incumplimiento menores (18.0%, 16.4% y
15.7%) que `2` (55.9%); `-2` concentra 55.4% de facturas no positivas y
mediana de factura 0, mientras que `0` tiene 1.8% de facturas no positivas y
mediana 48,703.5. La razón pago/factura vuelve a ser alta para `-2`/`-1`
(0.87/0.96) y baja para `0`/`2` (0.05). Que el patrón se repita en otro
periodo hace más consistente la lectura de que los códigos negativos/cero
se asocian a estados de facturación distintos.

**Qué NO se puede concluir.** El código `1` de `PAY_2` tiene solo 19 filas,
así que su tasa (15.8%) y su 73.7% de facturas no positivas son muy
inestables y no permiten generalizar. Sigue sin haber un diccionario oficial
para los códigos no documentados (`-2` y `0`): la repetición del patrón no
prueba el significado del código. También es una asociación descriptiva, no
causal, y `PAY_2`/`BILL_AMT2`/`PAY_AMT2` comparten el mismo historial de la
persona.

In [13]:
print("EDUCATION — códigos 0 a 6 (solo entrenamiento)")
tabla_significado_education = ev.tabla_categoria_significado(
    df_train,
    columna="EDUCATION",
    codigos=[0, 1, 2, 3, 4, 5, 6],
    objetivo=TARGET,
)
display(tabla_significado_education)

ruta_significado_education = RESULTADOS / "significado_education.csv"
tabla_significado_education.to_csv(
    ruta_significado_education, index=False, float_format="%.6f"
)
print(f"Guardado en: {ruta_significado_education}")

EDUCATION — códigos 0 a 6 (solo entrenamiento)


,variable,codigo,n,incumplimientos,tasa_incumplimiento,ic95_inferior,ic95_superior,edad_media,limit_bal_mediana,n_mujeres,pct_mujeres
0,EDUCATION,0,10,0,0.000000,0.000000,27.753280,40.600000,220000.0,5,50.000000
1,EDUCATION,1,7465,1431,19.169457,18.292456,20.078173,34.201474,200000.0,4396,58.888145
2,EDUCATION,2,9853,2353,23.881051,23.049479,24.732983,34.737542,110000.0,6075,61.656348
3,EDUCATION,3,3378,840,24.866785,23.438263,26.352405,40.327709,80000.0,2025,59.946714
4,EDUCATION,4,74,5,6.756757,2.920409,14.861194,34.040541,200000.0,47,63.513514
5,EDUCATION,5,194,10,5.154639,2.823678,9.227112,35.685567,150000.0,133,68.556701
6,EDUCATION,6,26,6,23.076923,11.033849,42.051554,41.538462,90000.0,15,57.692308


Guardado en: /home/tito/repos/RDP/Examen_1_RDP/resultados/significado_education.csv


**Qué sugiere.** Los códigos `0`, `4`, `5` y `6` tienen tasas de
incumplimiento bajas (0.0%, 6.8%, 5.2% y 23.1%), pero sus muestras son
pequeñas: 10, 74, 194 y 26 filas. Por eso sus intervalos de Wilson son
anchos: 0–27.8%, 2.9–14.9%, 2.8–9.2% y 11.0–42.1%. Los códigos `1`, `2` y
`3`, con miles de filas, tienen tasas de 19.2%, 23.9% y 24.9% e intervalos
mucho más estrechos. También cambian la edad media (34.0–41.5 años), la
mediana de `LIMIT_BAL` (80,000–220,000) y la proporción de mujeres
(50.0–68.6%), lo que sugiere que los códigos raros no son solo "otro nivel"
de educación, sino grupos con composición distinta.

**Qué NO se puede concluir.** Con `n` tan chico, las tasas de `0`, `4`, `5`
y `6` son compatibles con muchos valores dentro de su intervalo; no se
puede afirmar que su tasa "real" sea menor. No hay un diccionario oficial
que explique con precisión los códigos `0`, `5` y `6` (el `4` aparece como
"others"), y las diferencias son asociaciones, no efectos causales. No se
puede concluir que agrupar, eliminar o dejar como categoría propia sea
mejor: esta tabla solo describe.

In [14]:
print("MARRIAGE — códigos 0 a 3 (solo entrenamiento)")
tabla_significado_marriage = ev.tabla_categoria_significado(
    df_train,
    columna="MARRIAGE",
    codigos=[0, 1, 2, 3],
    objetivo=TARGET,
)
display(tabla_significado_marriage)

ruta_significado_marriage = RESULTADOS / "significado_marriage.csv"
tabla_significado_marriage.to_csv(
    ruta_significado_marriage, index=False, float_format="%.6f"
)
print(f"Guardado en: {ruta_significado_marriage}")

MARRIAGE — códigos 0 a 3 (solo entrenamiento)


,variable,codigo,n,incumplimientos,tasa_incumplimiento,ic95_inferior,ic95_superior,edad_media,limit_bal_mediana,n_mujeres,pct_mujeres
0,MARRIAGE,0,41,5,12.195122,5.323336,25.544212,37.658537,150000.0,32,78.048780
1,MARRIAGE,1,9522,2223,23.345936,22.507104,24.206265,39.952846,160000.0,5937,62.350347
2,MARRIAGE,2,11187,2349,20.997586,20.252868,21.762216,31.485027,130000.0,6578,58.800393
3,MARRIAGE,3,250,68,27.200000,22.060023,33.030056,42.160000,60000.0,149,59.600000


Guardado en: /home/tito/repos/RDP/Examen_1_RDP/resultados/significado_marriage.csv


**Qué sugiere.** El código `0` tiene 41 filas, una tasa de incumplimiento
de 12.2% (IC 95%: 5.3–25.5%), 78.0% de mujeres, edad media 37.7 años y
mediana de `LIMIT_BAL` de 150,000. Los códigos `1`, `2` y `3` tienen tasas
de 23.3%, 21.0% y 27.2%; el código `3` tiene solo 250 filas y un intervalo
de 22.1–33.0%, que se traslapa con los códigos `1` y `2`. La composición
por sexo y el límite de crédito también cambian entre códigos.

**Qué NO se puede concluir.** El código `0` tiene pocas filas y un
intervalo ancho; no se puede afirmar que su tasa sea realmente menor que la
de los códigos documentados. El código `3` también tiene un intervalo que
se traslapa con `1` y `2`, así que no hay separación clara. No hay
diccionario oficial para el `0`, y las diferencias son asociaciones, no
causalidad. Esta tabla no decide si el `0` debe agruparse, eliminarse o
conservarse.

## 4. Variantes con validación cruzada de 5 particiones

Se evalúan las 12 combinaciones: 3 variantes de códigos raros × 2 variantes
de `PAY_*` × 2 modelos. Todas usan las mismas 5 particiones estratificadas
del 70% de entrenamiento, generadas con `SEED`.

- **Códigos raros A:** agrupa `EDUCATION {0,5,6} -> 4` y `MARRIAGE {0} -> 3`.
- **Códigos raros B:** elimina esas filas solo del entrenamiento de cada
  fold; la validación conserva todas las filas para que los AUC sean
  comparables.
- **Códigos raros C:** deja los códigos raros como categorías propias
  (one-hot).
- **PAY_* A:** usa los valores numéricos tal cual.
- **PAY_* B:** crea indicadores para `-2`, `-1`, `0` y una columna de meses
  de atraso para valores `>= 1`.

Cada modelo se ajusta dentro de un `Pipeline` nuevo en cada fold. No se
buscan hiperparámetros: se usan los valores por defecto de scikit-learn,
salvo `max_iter` en la regresión logística para asegurar convergencia.

In [15]:
folds = ev.crear_folds(X_train, y_train, n_splits=5, seed=SEED)
tabla_combinaciones = ev.evaluar_combinaciones(
    X_train,
    y_train,
    folds,
    columnas_pay,
    seed=SEED,
)
display(tabla_combinaciones)

,experimento,variante_raros_id,variante_raros,variante_pay_id,variante_pay,modelo_id,modelo,con_sexo,auc_medio,auc_std,n_train_promedio
0,combinaciones,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.726661,0.006518,16800.0
1,combinaciones,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778908,0.006115,16800.0
2,combinaciones,A,A: agrupar códigos raros,B,"B: estados (-2, -1, 0) + meses de atraso",logistica,Regresión logística con escalado,sí,0.768654,0.002934,16800.0
3,combinaciones,A,A: agrupar códigos raros,B,"B: estados (-2, -1, 0) + meses de atraso",hist_gb,HistGradientBoostingClassifier,sí,0.778231,0.006270,16800.0
4,combinaciones,B,B: eliminar filas raras del entrenamiento,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.725092,0.006374,16583.2
5,combinaciones,B,B: eliminar filas raras del entrenamiento,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.777991,0.006165,16583.2
6,combinaciones,B,B: eliminar filas raras del entrenamiento,B,"B: estados (-2, -1, 0) + meses de atraso",logistica,Regresión logística con escalado,sí,0.767127,0.003544,16583.2
7,combinaciones,B,B: eliminar filas raras del entrenamiento,B,"B: estados (-2, -1, 0) + meses de atraso",hist_gb,HistGradientBoostingClassifier,sí,0.778570,0.006352,16583.2
8,combinaciones,C,C: one-hot de categorías propias,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.726532,0.006169,16800.0
9,combinaciones,C,C: one-hot de categorías propias,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778757,0.004750,16800.0


La tabla muestra el AUC medio y su desviación estándar para cada combinación.
La desviación estándar se calcula sobre los 5 valores de AUC de las
particiones (`ddof=0`). No se declara una ganadora: las diferencias entre
variantes deben compararse con la dispersión entre folds. Una desviación
estándar grande indica que el resultado depende de la partición; una
diferencia pequeña entre variantes no justifica por sí sola una decisión.

## 5. Comparación con y sin SEX

Para continuar con la comparación solicitada se toma la combinación con el
**mayor AUC medio** de la tabla anterior. Esto es solo un criterio empírico
para elegir sobre qué combinación repetir el experimento; no es una
recomendación de modelo ni una decisión sobre los datos.

In [16]:
mejor_idx = tabla_combinaciones["auc_medio"].idxmax()
mejor = tabla_combinaciones.loc[mejor_idx]
tabla_combinaciones["es_mejor_auc"] = False
tabla_combinaciones.loc[mejor_idx, "es_mejor_auc"] = True

print("Combinación con mayor AUC medio (solo para el paso 4):")
display(mejor.to_frame().T)

tabla_sexo, diferencia_auc = ev.evaluar_sexo(
    X_train,
    y_train,
    folds,
    mejor,
    columnas_pay,
    seed=SEED,
)
tabla_sexo["es_mejor_auc"] = False

display(tabla_sexo)
print(f"Diferencia de AUC (con SEX - sin SEX) = {diferencia_auc:.6f}")

Combinación con mayor AUC medio (solo para el paso 4):


,experimento,variante_raros_id,variante_raros,variante_pay_id,variante_pay,modelo_id,modelo,con_sexo,auc_medio,auc_std,n_train_promedio
1,combinaciones,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778908,0.006115,16800.0


,experimento,variante_raros_id,variante_raros,variante_pay_id,variante_pay,modelo_id,modelo,con_sexo,auc_medio,auc_std,n_train_promedio,diferencia_auc_con_menos_sin,es_mejor_auc
0,sexo,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778908,0.006115,16800.0,0.000129,False
1,sexo,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,no,0.778779,0.006701,16800.0,0.000129,False


Diferencia de AUC (con SEX - sin SEX) = 0.000129


La diferencia de AUC cuantifica cuánto cambia el desempeño al quitar `SEX`
de las entradas, manteniendo la misma combinación y las mismas particiones.
Un valor positivo indica que el AUC fue mayor al incluir `SEX`; uno negativo,
que fue mayor al excluirlo. Esa diferencia debe compararse con la desviación
estándar entre folds: si es mucho menor, no hay evidencia de una mejora
clara. Esta libreta solo reporta la diferencia: no decide si `SEX` debe
quedarse.

## 6. Tabla final

Se concatenan las 12 combinaciones y las 2 filas de la comparación con/sin
`SEX`. La columna `es_mejor_auc` marca la combinación de mayor AUC medio.
El archivo se guarda en `resultados/decisiones_datos.csv`.

In [17]:
tabla_final = pd.concat(
    [tabla_combinaciones, tabla_sexo],
    ignore_index=True,
    sort=False,
)
ruta_final = RESULTADOS / "decisiones_datos.csv"
tabla_final.to_csv(ruta_final, index=False, float_format="%.6f")

print(f"Tabla final guardada en: {ruta_final}")
display(tabla_final)

Tabla final guardada en: /home/tito/repos/RDP/Examen_1_RDP/resultados/decisiones_datos.csv


,experimento,variante_raros_id,variante_raros,variante_pay_id,variante_pay,modelo_id,modelo,con_sexo,auc_medio,auc_std,n_train_promedio,es_mejor_auc,diferencia_auc_con_menos_sin
0,combinaciones,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.726661,0.006518,16800.0,False,NaN
1,combinaciones,A,A: agrupar códigos raros,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778908,0.006115,16800.0,True,NaN
2,combinaciones,A,A: agrupar códigos raros,B,"B: estados (-2, -1, 0) + meses de atraso",logistica,Regresión logística con escalado,sí,0.768654,0.002934,16800.0,False,NaN
3,combinaciones,A,A: agrupar códigos raros,B,"B: estados (-2, -1, 0) + meses de atraso",hist_gb,HistGradientBoostingClassifier,sí,0.778231,0.006270,16800.0,False,NaN
4,combinaciones,B,B: eliminar filas raras del entrenamiento,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.725092,0.006374,16583.2,False,NaN
5,combinaciones,B,B: eliminar filas raras del entrenamiento,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.777991,0.006165,16583.2,False,NaN
6,combinaciones,B,B: eliminar filas raras del entrenamiento,B,"B: estados (-2, -1, 0) + meses de atraso",logistica,Regresión logística con escalado,sí,0.767127,0.003544,16583.2,False,NaN
7,combinaciones,B,B: eliminar filas raras del entrenamiento,B,"B: estados (-2, -1, 0) + meses de atraso",hist_gb,HistGradientBoostingClassifier,sí,0.778570,0.006352,16583.2,False,NaN
8,combinaciones,C,C: one-hot de categorías propias,A,A: PAY_* numérico tal cual,logistica,Regresión logística con escalado,sí,0.726532,0.006169,16800.0,False,NaN
9,combinaciones,C,C: one-hot de categorías propias,A,A: PAY_* numérico tal cual,hist_gb,HistGradientBoostingClassifier,sí,0.778757,0.004750,16800.0,False,NaN


La tabla final reúne la evidencia de las 12 combinaciones y del contraste
con/sin `SEX`. **No contiene una recomendación**: la decisión sobre qué
variante usar, si eliminar filas o si conservar `SEX` queda en manos del
alumno. Cualquier lectura de la tabla debe considerar el AUC medio junto con
su desviación estándar entre folds.